# Старт-2. Матрица попарных расстояний: циклы против векторизации

Сегодня одна задача, решённая тремя способами. Даны два набора точек: $A$ — $n$ точек, $B$ — $m$ точек, у каждой точки $d$ координат. Нужна матрица $D$ формы $(n, m)$, где

$$D_{ij} = \lVert a_i - b_j \rVert = \sqrt{\sum_{k=1}^{d} (a_{ik} - b_{jk})^2}.$$

Эта функция понадобится в неделе 11: классификатор kNN (CS231n, `knn.ipynb`) ищет ближайших соседей именно по такой матрице.

**Правила:** сначала думай сам. Застрял на 5 минут — перечитай подсказку в ячейке выше, потом документацию NumPy. Где застрял и что помогло — в `stuck.md`.

In [1]:
import numpy as np
import time
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## Маленький пример, который можно проверить руками

Прежде чем писать код, посчитай ответ **на бумаге**:

$A = \begin{pmatrix}0 & 0\\ 3 & 4\end{pmatrix}$, $B = \begin{pmatrix}0 & 0\\ 0 & 4\end{pmatrix}$.

Запиши свою матрицу $D$ в `D_small` (форма $(2, 2)$). Проверка ниже сравнит её с правильным ответом через контрольную сумму — сам ответ здесь не написан.

In [4]:
D_small = [[0, 4], 
           [5, 3]]
A_small = np.array([[0, 0], [3, 4]], dtype=float)
B_small = np.array([[0, 0], [0, 4]], dtype=float)

assert D_small is not None and np.shape(D_small) == (2, 2), "нужна матрица 2×2"
assert abs(np.sum(np.asarray(D_small) * np.array([[1, 2], [3, 4]])) - 35) < 1e-9, "пересчитай: какое-то расстояние неверно"

## Способ 1. Два цикла

Самый прямой перевод формулы: цикл по точкам $A$, внутри — цикл по точкам $B$.

**Напиши** `dist_two_loops(A, B)`. Для одной пары точек можно пользоваться NumPy (разность двух строк, сумма квадратов) — циклом по координатам $k$ писать не нужно.

Подумай: какой формы массив создать заранее и чем его заполнить?

In [ ]:
def dist_two_loops(A, B):
    solution = []
    for i in range(len(A)):
        helper = []
        for j in range(len(B)):
            helper.append('0')
        solution.append(helper)
    for i in range(len(A)):
        for j in range(len(B)):
            sum = 0
            for k in range(len(A[0])):
                sum += (A[i][k] - B[j][k])**2
            solution[i][j] = sum**0.5
    return np.array(solution)


assert np.allclose(dist_two_loops(A_small, B_small), D_small)

## Способ 2. Один цикл

Оставь только цикл по точкам $A$. За один шаг нужно посчитать расстояния от $a_i$ **сразу до всех** точек $B$ — то есть целую строку $D_{i,:}$.

Вопросы, на которые стоит ответить до кода:
- `A[i]` имеет форму $(d,)$, `B` — форму $(m, d)$. Какая форма будет у `A[i] - B` и почему это вообще работает? (Правило broadcasting из сегодняшнего чтения.)
- По какой оси суммировать квадраты, чтобы получить $m$ чисел?

**Напиши** `dist_one_loop(A, B)`.

In [ ]:
def dist_one_loop(A, B):
    n = A.shape[0]
    m = B.shape[0]
    solution = np.zeros((n, m))
    for i in range(n):
        solution[i] = np.sqrt(((A[i] - B) ** 2).sum(axis=1))
    return np.array(solution)

print(dist_one_loop(A_small, B_small))

# assert np.allclose(dist_one_loop(A_small, B_small), D_small)

[[0. 4.]
 [5. 3.]]


## Способ 3. Без циклов

Раскрой квадрат разности (это обычная алгебра, как $(x-y)^2$):

$$\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2\,a\cdot b.$$

Теперь каждое слагаемое нужно посчитать **для всех пар сразу**:

| Слагаемое | Для всех пар — это… | Форма |
|---|---|---|
| $\lVert a_i\rVert^2$ | квадрат длины каждой строки $A$ | $(n,)$ |
| $\lVert b_j\rVert^2$ | квадрат длины каждой строки $B$ | $(m,)$ |
| $a_i \cdot b_j$ | все скалярные произведения — одна матричная операция. Какая? | $(n, m)$ |

Дальше сложи три части в матрицу $(n, m)$. Подумай, какой из векторов надо сделать **столбцом** $(n, 1)$, а какой оставить **строкой** $(1, m)$, чтобы broadcasting дал нужную форму.

**Ловушка:** из-за округления вместо точного нуля под корнем иногда получается $-10^{-15}$, и `np.sqrt` возвращает `nan`. Что с этим сделать перед корнем?

**Напиши** `dist_no_loops(A, B)` — ни одного `for`.

In [19]:
def dist_no_loops(A, B):
    a2 = (A**2).sum(axis=1)[:, None]
    b2 = (B**2).sum(axis=1)
    ab = A @ B.T
    squared = a2 + b2 - 2 * ab
    squared = np.maximum(squared, 0)
    return np.sqrt(squared)

assert np.allclose(dist_no_loops(A_small, B_small), D_small)
assert not np.isnan(dist_no_loops(A_small, A_small)).any(), "расстояние точки до самой себя дало nan — вспомни про ловушку"

## Проверка на случайных данных

Маленький пример мог совпасть случайно. Сравни все три версии на случайных точках разных размеров (включая $n \ne m$).

In [24]:
for n, m, d in [(1, 1, 1), (5, 7, 3), (100, 80, 64)]:
    A = rng.standard_normal((n, d))
    B = rng.standard_normal((m, d))
    D1, D2, D3 = dist_two_loops(A, B), dist_one_loop(A, B), dist_no_loops(A, B)
    assert D1.shape == (n, m)
    assert np.allclose(D1, D2) and np.allclose(D1, D3), f"версии расходятся при n={n}, m={m}, d={d}"
print("Все три версии совпадают")

AttributeError: 'list' object has no attribute 'shape'

## Замер времени

Замерь каждую версию на $n = m = 2000$, $d = 64$.

- Версии 2 и 3 — обычным `%timeit`.
- Версия 1 — это 4 миллиона итераций Python, а `%timeit` по умолчанию повторяет замер много раз. Запусти её **один раз**: `%timeit -n1 -r1 ...`. Если ждать дольше пары минут — остановись и замерь на $n = m = 500$.

Прежде чем запускать: **предскажи**, во сколько раз версия 3 будет быстрее версии 1. Запиши предсказание в комментарий, потом сравни.

In [ ]:
A = rng.standard_normal((2000, 64))
B = rng.standard_normal((2000, 64))

# Предсказание: версия 3 быстрее версии 1 примерно в ... раз
# %timeit dist_no_loops(A, B)
# %timeit dist_one_loop(A, B)
# %timeit -n1 -r1 dist_two_loops(A, B)

## График времени от $n$

**Напиши** цикл по размерам, например $n = m \in \{100, 200, 500, 1000, 2000\}$, и замерь каждую версию через `time.perf_counter()` (время «до» и «после»). Версию с двумя циклами мерь только до 500–1000, иначе ждать слишком долго.

Построй **три линии на одном графике**: по оси $x$ — $n$, по оси $y$ — время в секундах. Подпиши оси, добавь легенду. Какую шкалу поставить по $y$, чтобы быстрая версия не слилась с нулём?

In [ ]:
sizes = [100, 200, 500, 1000, 2000]

# 1) замерь время каждой версии для каждого n
# 2) fig, ax = plt.subplots() — три линии, подписи осей, легенда, подходящая шкала по y


## Вопросы для journal.md

Ответь своими словами (2–3 предложения на каждый):

1. Почему версия без циклов быстрее, хотя делает те же вычисления?
2. Как растёт время с ростом $n$ — примерно как $n$, $n^2$ или $n^3$? Видно ли это по наклону линий на лог-графике?
3. Зачем обрезать значения нулём перед корнем?
4. У версии 3 есть минус по памяти: какие промежуточные массивы размера $(n, m)$ она создаёт? Что будет при $n = m = 10^5$?

**Чекпоинт дня:** три версии совпадают, есть график и цифры ускорения, ответы записаны.